# Hands-on Exercise 2 — House Price Prediction (Data Preparation)

**DigiSkills AI Using Python** — The next cell looks for the LMS file in this order: `HousePricePrediction.csv` next to this notebook, then `HousePricePrediction.csv` or `HousePricePrediction (2).csv` in your **Downloads** folder (browsers often add `(2)` when you download again).

Use each numbered section for your MS Word screenshots (code + output).

## 1. Dataset Loading (Marks 2)
- Load the dataset into a Pandas DataFrame
- Display the first 10 rows

In [9]:
import pandas as pd
from pathlib import Path

DATA_CANDIDATES = [
    Path.cwd() / "HousePricePrediction.csv",
    Path.home() / "Downloads" / "HousePricePrediction.csv",
    Path.home() / "Downloads" / "HousePricePrediction (2).csv",
]
DATA_FILE = next((p for p in DATA_CANDIDATES if p.is_file()), None)
if DATA_FILE is None:
    raise FileNotFoundError(
        "Could not find the CSV. Save it as HousePricePrediction.csv next to this notebook, "
        "or as HousePricePrediction.csv / HousePricePrediction (2).csv in your Downloads folder."
    )

df = pd.read_csv(DATA_FILE)
print("Loaded from:", DATA_FILE.resolve())
df.head(10)

Loaded from: C:\Users\DELL\Downloads\HousePricePrediction.csv


,Id,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice
0,0,60,RL,8450,Inside,1Fam,5,2003,2003,VinylSd,0.0,856.0,208500.0
1,1,20,RL,9600,FR2,1Fam,8,1976,1976,MetalSd,0.0,1262.0,181500.0
2,2,60,RL,11250,Inside,1Fam,5,2001,2002,VinylSd,0.0,920.0,223500.0
3,3,70,RL,9550,Corner,1Fam,5,1915,1970,Wd Sdng,0.0,756.0,140000.0
4,4,60,RL,14260,FR2,1Fam,5,2000,2000,VinylSd,0.0,1145.0,250000.0
5,5,50,RL,14115,Inside,1Fam,5,1993,1995,VinylSd,0.0,796.0,143000.0
6,6,20,RL,10084,Inside,1Fam,5,2004,2005,VinylSd,0.0,1686.0,307000.0
7,7,60,RL,10382,Corner,1Fam,6,1973,1973,HdBoard,32.0,1107.0,200000.0
8,8,50,RM,6120,Inside,1Fam,5,1931,1950,BrkFace,0.0,952.0,129900.0
9,9,190,RL,7420,Corner,2fmCon,6,1939,1950,MetalSd,0.0,991.0,118000.0


## 2. Data Exploration (Marks 2)
- Shape of the dataset
- Data types of all columns
- Summary statistics of numerical features

In [11]:
print("Shape (rows, columns):", df.shape)
print()
print("Data types:")
print(df.dtypes)
print()
print("Summary statistics (numerical columns):")
df.describe()

Shape (rows, columns): (2919, 13)

Data types:
Id                int64
MSSubClass        int64
MSZoning            str
LotArea           int64
LotConfig           str
BldgType            str
OverallCond       int64
YearBuilt         int64
YearRemodAdd      int64
Exterior1st         str
BsmtFinSF2      float64
TotalBsmtSF     float64
SalePrice       float64
dtype: object

Summary statistics (numerical columns):


,Id,MSSubClass,LotArea,OverallCond,YearBuilt,YearRemodAdd,BsmtFinSF2,TotalBsmtSF,SalePrice
count,2919.000000,2919.000000,2919.000000,2919.000000,2919.000000,2919.000000,2918.000000,2918.000000,1460.000000
mean,1459.000000,57.137718,10168.114080,5.564577,1971.312778,1984.264474,49.582248,1051.777587,180921.195890
std,842.787043,42.517628,7886.996359,1.113131,30.291442,20.894344,169.205611,440.766258,79442.502883
min,0.000000,20.000000,1300.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,34900.000000
25%,729.500000,20.000000,7478.000000,5.000000,1953.500000,1965.000000,0.000000,793.000000,129975.000000
50%,1459.000000,50.000000,9453.000000,5.000000,1973.000000,1993.000000,0.000000,989.500000,163000.000000
75%,2188.500000,70.000000,11570.000000,6.000000,2001.000000,2004.000000,0.000000,1302.000000,214000.000000
max,2918.000000,190.000000,215245.000000,9.000000,2010.000000,2010.000000,1526.000000,6110.000000,755000.000000


## 3. Data Cleaning (Marks 2)
- Identify missing values
- Handle missing values appropriately
- Check and remove duplicate records if any

In [17]:
missing_per_column = df.isna().sum()
missing_nonzero = missing_per_column[missing_per_column > 0].sort_values(ascending=False)
print("Missing value counts (columns with at least one missing value):")
print(missing_nonzero if len(missing_nonzero) else "No missing values.")
print("\nTotal missing cells:", int(df.isna().sum().sum()))

Missing value counts (columns with at least one missing value):
SalePrice      1459
MSZoning          4
Exterior1st       1
BsmtFinSF2        1
TotalBsmtSF       1
dtype: int64

Total missing cells: 1466


In [13]:
df_clean = df.copy()

num_cols = df_clean.select_dtypes(include=["number"]).columns
cat_cols = df_clean.select_dtypes(exclude=["number"]).columns

for c in num_cols:
    if df_clean[c].isna().any():
        df_clean[c] = df_clean[c].fillna(df_clean[c].median())

for c in cat_cols:
    if df_clean[c].isna().any():
        mode = df_clean[c].mode(dropna=True)
        fill = mode.iloc[0] if len(mode) else "Unknown"
        df_clean[c] = df_clean[c].fillna(fill)

print("Missing values after imputation:", int(df_clean.isna().sum().sum()))

dup_before = df_clean.duplicated().sum()
print("Duplicate rows (before removal):", int(dup_before))
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df_clean.shape)

Missing values after imputation: 0
Duplicate rows (before removal): 0
Shape after removing duplicates: (2919, 13)


## 4. Feature Selection (Marks 2)
- Identify relevant features for analysis
- Remove unnecessary identifier columns (e.g. `Id`)

We keep property and sale-related attributes and drop `Id`, which is not predictive.

In [19]:
cols_to_drop = [c for c in ["Id", "id"] if c in df_clean.columns]
df_features = df_clean.drop(columns=cols_to_drop, errors="ignore")

print("Dropped columns:", cols_to_drop if cols_to_drop else "(none named Id/id)")
print("Remaining columns:", df_features.shape[1])
print(list(df_features.columns[:15]), "...")

Dropped columns: ['Id']
Remaining columns: 12
['MSSubClass', 'MSZoning', 'LotArea', 'LotConfig', 'BldgType', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'Exterior1st', 'BsmtFinSF2', 'TotalBsmtSF', 'SalePrice'] ...


## 5. Data Preprocessing (Marks 2)
- Encode categorical variables as numbers (one-hot encoding)
- Preview the dataset ready for machine learning

In [20]:
X_cat = df_features.select_dtypes(include=["object", "category"]).columns
df_ml = pd.get_dummies(df_features, columns=list(X_cat), drop_first=True, dtype=int)

print("ML-ready shape:", df_ml.shape)
print("All columns are numeric:", df_ml.select_dtypes(exclude=["number"]).shape[1] == 0)
df_ml.head(10)

ML-ready shape: (2919, 34)
All columns are numeric: True


C:\Users\DELL\AppData\Local\Temp\ipykernel_6868\2343590877.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X_cat = df_features.select_dtypes(include=["object", "category"]).columns


,MSSubClass,LotArea,OverallCond,YearBuilt,YearRemodAdd,BsmtFinSF2,TotalBsmtSF,SalePrice,MSZoning_FV,MSZoning_RH,...,Exterior1st_CemntBd,Exterior1st_HdBoard,Exterior1st_ImStucc,Exterior1st_MetalSd,Exterior1st_Plywood,Exterior1st_Stone,Exterior1st_Stucco,Exterior1st_VinylSd,Exterior1st_Wd Sdng,Exterior1st_WdShing
0,60,8450,5,2003,2003,0.0,856.0,208500.0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,20,9600,8,1976,1976,0.0,1262.0,181500.0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,60,11250,5,2001,2002,0.0,920.0,223500.0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,70,9550,5,1915,1970,0.0,756.0,140000.0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,60,14260,5,2000,2000,0.0,1145.0,250000.0,0,0,...,0,0,0,0,0,0,0,1,0,0
5,50,14115,5,1993,1995,0.0,796.0,143000.0,0,0,...,0,0,0,0,0,0,0,1,0,0
6,20,10084,5,2004,2005,0.0,1686.0,307000.0,0,0,...,0,0,0,0,0,0,0,1,0,0
7,60,10382,6,1973,1973,32.0,1107.0,200000.0,0,0,...,0,1,0,0,0,0,0,0,0,0
8,50,6120,5,1931,1950,0.0,952.0,129900.0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,190,7420,6,1939,1950,0.0,991.0,118000.0,0,0,...,0,0,0,1,0,0,0,0,0,0


In [ ]:
# save cleaned + encoded data for Exercise 3 :

OUTPUT_CSV = "HousePricePrediction_ml_ready.csv"
df_ml.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

Saved: HousePricePrediction_ml_ready.csv
